<a href="https://colab.research.google.com/github/sayamniper2024-maker/medtech-regulatory-navigator/blob/main/sam_medtech_navigator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
# Run this first — installs all required libraries into Colab's environment
!pip install streamlit pandas plotly openpyxl -q

print("✅ All libraries installed!")

✅ All libraries installed!


In [48]:
import pandas as pd
import os

# Create a data folder inside Colab's file system
os.makedirs("data", exist_ok=True)

# ── CDSCO Rules (India - MDR 2017) ──────────────────────────────────────────
cdsco_data = {
    "device_type": [
        "Thermometer", "Blood Pressure Monitor", "Pulse Oximeter",
        "ECG Machine", "Ventilator", "Pacemaker",
        "Surgical Scissors", "Infusion Pump", "MRI Scanner",
        "HIV Test Kit", "Glucose Meter", "Bone Implant"
    ],
    "risk_class": ["A", "B", "B", "C", "C", "D", "A", "C", "C", "C", "B", "D"],
    "intended_use": [
        "Temperature measurement", "BP monitoring", "Oxygen saturation",
        "Cardiac monitoring", "Life support", "Cardiac rhythm management",
        "Surgical cutting", "Drug delivery", "Diagnostic imaging",
        "Disease diagnosis", "Glucose monitoring", "Bone replacement"
    ],
    "license_type": [
        "MD-5", "MD-5", "MD-5", "MD-9", "MD-9", "MD-14",
        "MD-5", "MD-9", "MD-9", "MD-9", "MD-5", "MD-14"
    ],
    "timeline_months": [3, 6, 6, 9, 9, 18, 3, 9, 9, 9, 6, 18],
    "qms_required": [
        "No", "Yes", "Yes", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "Yes", "Yes", "Yes"
    ],
    "clinical_data_required": [
        "No", "No", "No", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "Yes", "No", "Yes"
    ]
}

# ── FDA Rules (USA - 510k) ───────────────────────────────────────────────────
fda_data = {
    "device_type": [
        "Thermometer", "Blood Pressure Monitor", "Pulse Oximeter",
        "ECG Machine", "Ventilator", "Pacemaker",
        "Surgical Scissors", "Infusion Pump", "MRI Scanner",
        "HIV Test Kit", "Glucose Meter", "Bone Implant"
    ],
    "risk_class": ["I", "II", "II", "II", "III", "III", "I", "II", "II", "III", "II", "III"],
    "pathway": [
        "Exempt", "510(k)", "510(k)", "510(k)", "PMA", "PMA",
        "Exempt", "510(k)", "510(k)", "PMA", "510(k)", "PMA"
    ],
    "predicate_needed": [
        "No", "Yes", "Yes", "Yes", "No", "No",
        "No", "Yes", "Yes", "No", "Yes", "No"
    ],
    "timeline_months": [1, 6, 6, 6, 36, 36, 1, 9, 9, 36, 6, 36],
    "clinical_trials_required": [
        "No", "No", "No", "No", "Yes", "Yes",
        "No", "No", "No", "Yes", "No", "Yes"
    ],
    "ide_required": [
        "No", "No", "No", "No", "Yes", "Yes",
        "No", "No", "No", "Yes", "No", "Yes"
    ]
}

# ── EU MDR Rules (CE Mark) ───────────────────────────────────────────────────
eu_data = {
    "device_type": [
        "Thermometer", "Blood Pressure Monitor", "Pulse Oximeter",
        "ECG Machine", "Ventilator", "Pacemaker",
        "Surgical Scissors", "Infusion Pump", "MRI Scanner",
        "HIV Test Kit", "Glucose Meter", "Bone Implant"
    ],
    "risk_class": ["I", "IIa", "IIa", "IIb", "IIb", "III", "I", "IIb", "IIa", "III", "IIa", "III"],
    "notified_body_needed": [
        "No", "Yes", "Yes", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "Yes", "Yes", "Yes"
    ],
    "timeline_months": [3, 9, 9, 12, 18, 24, 3, 12, 12, 18, 9, 24],
    "technical_file_type": [
        "Basic UDI-DI", "Full Tech File", "Full Tech File",
        "Full Tech File", "Full Tech File", "Full Tech File",
        "Basic UDI-DI", "Full Tech File", "Full Tech File",
        "Full Tech File", "Full Tech File", "Full Tech File"
    ],
    "clinical_evaluation_required": [
        "No", "Yes", "Yes", "Yes", "Yes", "Yes",
        "No", "Yes", "Yes", "Yes", "Yes", "Yes"
    ],
    "pmcf_required": [
        "No", "No", "No", "Yes", "Yes", "Yes",
        "No", "Yes", "No", "Yes", "No", "Yes"
    ]
}

# ── Write all 3 sheets into one Excel file ───────────────────────────────────
with pd.ExcelWriter("data/regulatory_rules.xlsx", engine="openpyxl") as writer:
    pd.DataFrame(cdsco_data).to_excel(writer, sheet_name="CDSCO",  index=False)
    pd.DataFrame(fda_data).to_excel(writer,   sheet_name="FDA",    index=False)
    pd.DataFrame(eu_data).to_excel(writer,    sheet_name="EU_MDR", index=False)

print("✅ regulatory_rules.xlsx created in data/ folder!")
print(f"   Devices per framework: {len(cdsco_data['device_type'])}")

# Preview the CDSCO sheet so you can see it worked
print("\n--- CDSCO Preview ---")
print(pd.read_excel("data/regulatory_rules.xlsx", sheet_name="CDSCO").to_string(index=False))

✅ regulatory_rules.xlsx created in data/ folder!
   Devices per framework: 12

--- CDSCO Preview ---
           device_type risk_class              intended_use license_type  timeline_months qms_required clinical_data_required
           Thermometer          A   Temperature measurement         MD-5                3           No                     No
Blood Pressure Monitor          B             BP monitoring         MD-5                6          Yes                     No
        Pulse Oximeter          B         Oxygen saturation         MD-5                6          Yes                     No
           ECG Machine          C        Cardiac monitoring         MD-9                9          Yes                    Yes
            Ventilator          C              Life support         MD-9                9          Yes                    Yes
             Pacemaker          D Cardiac rhythm management        MD-14               18          Yes                    Yes
     Surgical Sci

In [49]:
# In Colab we define functions directly in cells instead of separate files
# This is equivalent to utils/classifier.py

df_cdsco = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="CDSCO")
df_fda   = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="FDA")
df_eu    = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="EU_MDR")

def get_all_devices():
    """Returns sorted list of all device names for dropdowns."""
    return sorted(df_cdsco["device_type"].tolist())

def classify_device(device_name):
    """
    Looks up a device and returns its risk class across all 3 frameworks.
    Think of this like a VLOOKUP across 3 Excel sheets at once.
    """
    cdsco_row = df_cdsco.loc[df_cdsco["device_type"] == device_name]
    fda_row   = df_fda.loc[df_fda["device_type"]     == device_name]
    eu_row    = df_eu.loc[df_eu["device_type"]        == device_name]

    if cdsco_row.empty:
        return {"found": False, "device": device_name}

    return {
        "found"       : True,
        "device"      : device_name,
        "cdsco_class" : cdsco_row.iloc[0]["risk_class"],
        "fda_class"   : fda_row.iloc[0]["risk_class"],
        "eu_class"    : eu_row.iloc[0]["risk_class"],
        "intended_use": cdsco_row.iloc[0]["intended_use"]
    }

def get_risk_level(risk_class):
    """Converts a class code like 'D' or 'III' into Low/Medium/High/Critical."""
    risk_map = {
        "A": "Low",    "B": "Medium", "C": "High",   "D": "Critical",
        "I": "Low",    "II": "Medium","III": "Critical",
        "IIa": "Medium", "IIb": "High"
    }
    return risk_map.get(risk_class, "Unknown")

print("✅ Classifier functions defined!")
print("Available devices:", get_all_devices())

✅ Classifier functions defined!
Available devices: ['Blood Pressure Monitor', 'Bone Implant', 'ECG Machine', 'Glucose Meter', 'HIV Test Kit', 'Infusion Pump', 'MRI Scanner', 'Pacemaker', 'Pulse Oximeter', 'Surgical Scissors', 'Thermometer', 'Ventilator']


In [50]:
# Equivalent to utils/recommender.py

def get_cdsco_recommendation(device_name):
    row = df_cdsco.loc[df_cdsco["device_type"] == device_name]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "framework"             : "CDSCO (India)",
        "risk_class"            : r["risk_class"],
        "license_type"          : r["license_type"],
        "timeline_months"       : r["timeline_months"],
        "qms_required"          : r["qms_required"],
        "clinical_data_required": r["clinical_data_required"],
        "key_requirements"      : _build_cdsco_requirements(r)
    }

def get_fda_recommendation(device_name):
    row = df_fda.loc[df_fda["device_type"] == device_name]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "framework"               : "FDA 510(k) (USA)",
        "risk_class"              : r["risk_class"],
        "pathway"                 : r["pathway"],
        "predicate_needed"        : r["predicate_needed"],
        "timeline_months"         : r["timeline_months"],
        "clinical_trials_required": r["clinical_trials_required"],
        "ide_required"            : r["ide_required"],
        "key_requirements"        : _build_fda_requirements(r)
    }

def get_eu_recommendation(device_name):
    row = df_eu.loc[df_eu["device_type"] == device_name]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "framework"                   : "CE Mark (EU MDR)",
        "risk_class"                  : r["risk_class"],
        "notified_body_needed"        : r["notified_body_needed"],
        "timeline_months"             : r["timeline_months"],
        "technical_file_type"         : r["technical_file_type"],
        "clinical_evaluation_required": r["clinical_evaluation_required"],
        "pmcf_required"               : r["pmcf_required"],
        "key_requirements"            : _build_eu_requirements(r)
    }

def get_fastest_pathway(device_name):
    """Compares all 3 timelines and tells you which market is fastest to enter."""
    timelines = {
        "CDSCO (India)": get_cdsco_recommendation(device_name)["timeline_months"],
        "FDA (USA)"    : get_fda_recommendation(device_name)["timeline_months"],
        "CE Mark (EU)" : get_eu_recommendation(device_name)["timeline_months"]
    }
    return {"fastest_market": min(timelines, key=timelines.get), "timelines": timelines}

# ── Private helper functions ─────────────────────────────────────────────────
def _build_cdsco_requirements(r):
    reqs = [
        "Form MD-1 (Import/Manufacture license application)",
        "Device Master File",
        "ISO 13485 certificate" if r["qms_required"] == "Yes" else "Basic QMS documentation",
        "Test reports from NABL/BIS approved lab",
    ]
    if r["clinical_data_required"] == "Yes":
        reqs.append("Clinical performance data / PMCF report")
    if r["risk_class"] in ["C", "D"]:
        reqs.append("Free Sale Certificate from country of origin")
        reqs.append("Undertaking from manufacturer")
    return reqs

def _build_fda_requirements(r):
    reqs = [
        "Device description and intended use",
        "Substantial equivalence comparison" if r["predicate_needed"] == "Yes" else "Safety & effectiveness data",
        "Performance testing data",
        "Labeling (21 CFR Part 801)",
        "Quality System Regulation (21 CFR Part 820)",
    ]
    if r["clinical_trials_required"] == "Yes":
        reqs.append("Clinical trial data (IDE required)")
    if r["pathway"] == "PMA":
        reqs.append("PMA application — full safety dossier")
    return reqs

def _build_eu_requirements(r):
    reqs = [
        "Technical documentation (Annex II/III EU MDR)",
        "Declaration of Conformity",
        "UDI registration in EUDAMED",
        "Post-Market Surveillance (PMS) plan",
    ]
    if r["notified_body_needed"] == "Yes":
        reqs.append("Notified Body conformity assessment")
        reqs.append("QMS audit (ISO 13485 or Annex IX EU MDR)")
    if r["clinical_evaluation_required"] == "Yes":
        reqs.append("Clinical Evaluation Report (CER)")
    if r["pmcf_required"] == "Yes":
        reqs.append("Post-Market Clinical Follow-up (PMCF) plan")
    return reqs

print("✅ Recommender functions defined!")

✅ Recommender functions defined!


In [51]:
# Test device: Pacemaker (highest risk — good stress test)
test_device = "Pacemaker"

print("=" * 55)
print(f"  DEVICE: {test_device}")
print("=" * 55)

# Classification
classification = classify_device(test_device)
print(f"\n  CDSCO Class : {classification['cdsco_class']}  ({get_risk_level(classification['cdsco_class'])} risk)")
print(f"  FDA Class   : {classification['fda_class']}   ({get_risk_level(classification['fda_class'])} risk)")
print(f"  EU Class    : {classification['eu_class']}  ({get_risk_level(classification['eu_class'])} risk)")

# Recommendations
cdsco = get_cdsco_recommendation(test_device)
fda   = get_fda_recommendation(test_device)
eu    = get_eu_recommendation(test_device)

print(f"\n  CDSCO → License: {cdsco['license_type']}, Timeline: {cdsco['timeline_months']} months")
print(f"  FDA   → Pathway: {fda['pathway']},  Timeline: {fda['timeline_months']} months")
print(f"  EU    → Timeline: {eu['timeline_months']} months")

# Fastest market
fastest = get_fastest_pathway(test_device)
print(f"\n  Fastest market entry: {fastest['fastest_market']}")
print(f"  All timelines: {fastest['timelines']}")

print("\n  CDSCO Requirements:")
for req in cdsco["key_requirements"]:
    print(f"    - {req}")

print("\n✅ All systems working! Ready for Phase 4 (Streamlit UI)")

  DEVICE: Pacemaker

  CDSCO Class : D  (Critical risk)
  FDA Class   : III   (Critical risk)
  EU Class    : III  (Critical risk)

  CDSCO → License: MD-14, Timeline: 18 months
  FDA   → Pathway: PMA,  Timeline: 36 months
  EU    → Timeline: 24 months

  Fastest market entry: CDSCO (India)
  All timelines: {'CDSCO (India)': np.int64(18), 'FDA (USA)': np.int64(36), 'CE Mark (EU)': np.int64(24)}

  CDSCO Requirements:
    - Form MD-1 (Import/Manufacture license application)
    - Device Master File
    - ISO 13485 certificate
    - Test reports from NABL/BIS approved lab
    - Clinical performance data / PMCF report
    - Free Sale Certificate from country of origin
    - Undertaking from manufacturer

✅ All systems working! Ready for Phase 4 (Streamlit UI)


In [52]:
# Colab can't open a browser tab directly, so we use a tunnel
# pyngrok creates a public URL that forwards to your Streamlit app

!pip install pyngrok -q

# Sign up free at https://ngrok.com → Dashboard → Your Authtoken → copy it
# Paste your token below replacing YOUR_TOKEN_HERE
from pyngrok import ngrok
from google.colab import userdata

# Retrieve the NGROK_AUTH_TOKEN from Colab secrets
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

print("✅ Tunnel ready!")

✅ Tunnel ready!


In [53]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ─────────────────────────────────────────────
# Page config — must be the FIRST streamlit call
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="MedTech Regulatory Navigator",
    page_icon="🧬",
    layout="wide"
)

# ─────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────
@st.cache_data
def load_data():
    df_cdsco = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="CDSCO")
    df_fda   = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="FDA")
    df_eu    = pd.read_excel("data/regulatory_rules.xlsx", sheet_name="EU_MDR")
    return df_cdsco, df_fda, df_eu

df_cdsco, df_fda, df_eu = load_data()

# ─────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────
def classify_device(device_name):
    cdsco_row = df_cdsco.loc[df_cdsco["device_type"] == device_name].iloc[0]
    fda_row   = df_fda.loc[df_fda["device_type"]     == device_name].iloc[0]
    eu_row    = df_eu.loc[df_eu["device_type"]        == device_name].iloc[0]
    return cdsco_row, fda_row, eu_row

def get_risk_level(risk_class):
    risk_map = {
        "A":"Low","B":"Medium","C":"High","D":"Critical",
        "I":"Low","II":"Medium","III":"Critical",
        "IIa":"Medium","IIb":"High"
    }
    return risk_map.get(risk_class, "Unknown")

def get_risk_color(level):
    return {"Low":"#1D9E75","Medium":"#BA7517","High":"#D85A30","Critical":"#E24B4A"}.get(level,"#888")

def build_requirements(row, framework):
    if framework == "CDSCO":
        reqs = ["Form MD-1 application","Device Master File",
                "ISO 13485 certificate" if row["qms_required"]=="Yes" else "Basic QMS docs",
                "NABL/BIS lab test reports"]
        if row["clinical_data_required"] == "Yes":
            reqs.append("Clinical performance data")
        if row["risk_class"] in ["C","D"]:
            reqs += ["Free Sale Certificate","Manufacturer undertaking"]
    elif framework == "FDA":
        reqs = ["Device description & intended use",
                "Predicate device comparison" if row["predicate_needed"]=="Yes" else "Safety & effectiveness data",
                "Performance testing data","Labeling (21 CFR Part 801)","QSR (21 CFR Part 820)"]
        if row["clinical_trials_required"] == "Yes":
            reqs.append("Clinical trial data + IDE application")
        if row["pathway"] == "PMA":
            reqs.append("Full PMA safety dossier")
    else:
        reqs = ["Technical documentation (Annex II/III)","Declaration of Conformity",
                "UDI registration in EUDAMED","PMS plan"]
        if row["notified_body_needed"] == "Yes":
            reqs += ["Notified Body assessment","ISO 13485 QMS audit"]
        if row["clinical_evaluation_required"] == "Yes":
            reqs.append("Clinical Evaluation Report (CER)")
        if row["pmcf_required"] == "Yes":
            reqs.append("PMCF plan")
    return reqs

# ─────────────────────────────────────────────
# UI — Header
# ─────────────────────────────────────────────
st.markdown("## 🧬 MedTech Regulatory Pathway Navigator")
st.markdown("*Instantly identify the optimal regulatory pathway across CDSCO, FDA, and CE Mark*")
st.divider()

# ─────────────────────────────────────────────
# Sidebar — inputs
# ─────────────────────────────────────────────
with st.sidebar:
    st.header("Device Selection")
    all_devices = sorted(df_cdsco["device_type"].tolist())
    selected_device = st.selectbox("Select Device Type", all_devices, index=5)

    st.subheader("Target Markets")
    show_cdsco = st.checkbox("India (CDSCO / MDR 2017)", value=True)
    show_fda   = st.checkbox("USA (FDA 510k / PMA)",     value=True)
    show_eu    = st.checkbox("Europe (CE Mark / EU MDR)", value=True)

    st.divider()
    analyse = st.button("Analyse Device", type="primary", use_container_width=True)
    st.caption("Select a device and markets, then click Analyse.")

# ─────────────────────────────────────────────
# Main panel — results
# ─────────────────────────────────────────────
if analyse or True:   # auto-show on load
    cdsco_row, fda_row, eu_row = classify_device(selected_device)

    # ── Risk class summary ──────────────────────────────────────────────────
    st.subheader(f"Risk Classification — {selected_device}")
    col1, col2, col3 = st.columns(3)

    with col1:
        lvl = get_risk_level(cdsco_row["risk_class"])
        st.metric("CDSCO Class (India)", cdsco_row["risk_class"])
        st.markdown(f"<span style='color:{get_risk_color(lvl)};font-weight:600'>{lvl} Risk</span>",
                    unsafe_allow_html=True)

    with col2:
        lvl = get_risk_level(fda_row["risk_class"])
        st.metric("FDA Class (USA)", fda_row["risk_class"])
        st.markdown(f"<span style='color:{get_risk_color(lvl)};font-weight:600'>{lvl} Risk</span>",
                    unsafe_allow_html=True)

    with col3:
        lvl = get_risk_level(eu_row["risk_class"])
        st.metric("EU MDR Class (Europe)", eu_row["risk_class"])
        st.markdown(f"<span style='color:{get_risk_color(lvl)};font-weight:600'>{lvl} Risk</span>",
                    unsafe_allow_html=True)

    st.divider()

    # ── Timeline comparison chart ────────────────────────────────────────────
    st.subheader("Approval Timeline Comparison")

    timelines = {
        "CDSCO (India)" : int(cdsco_row["timeline_months"]),
        "FDA (USA)"     : int(fda_row["timeline_months"]),
        "CE Mark (EU)"  : int(eu_row["timeline_months"])
    }

    active_markets   = []
    active_timelines = []
    active_colors    = []
    color_map = {"CDSCO (India)":"#1D9E75","FDA (USA)":"#378ADD","CE Mark (EU)":"#D85A30"}

    if show_cdsco:
        active_markets.append("CDSCO (India)")
        active_timelines.append(timelines["CDSCO (India)"])
        active_colors.append(color_map["CDSCO (India)"])
    if show_fda:
        active_markets.append("FDA (USA)")
        active_timelines.append(timelines["FDA (USA)"])
        active_colors.append(color_map["FDA (USA)"])
    if show_eu:
        active_markets.append("CE Mark (EU)")
        active_timelines.append(timelines["CE Mark (EU)"])
        active_colors.append(color_map["CE Mark (EU)"])

    if active_markets:
        fig = go.Figure(go.Bar(
            x=active_markets,
            y=active_timelines,
            marker_color=active_colors,
            text=[f"{t} months" for t in active_timelines],
            textposition="outside"
        ))
        fig.update_layout(
            yaxis_title="Months to approval",
            plot_bgcolor="white",
            height=350,
            margin=dict(t=20, b=20)
        )
        st.plotly_chart(fig, use_container_width=True)

        fastest = min(zip(active_markets, active_timelines), key=lambda x: x[1])
        st.success(f"Fastest market entry: **{fastest[0]}** at **{fastest[1]} months**")

    st.divider()

    # ── Detailed pathway cards ───────────────────────────────────────────────
    st.subheader("Detailed Pathway Requirements")
    cols = st.columns(sum([show_cdsco, show_fda, show_eu]))
    col_idx = 0

    if show_cdsco:
        with cols[col_idx]:
            st.markdown("#### 🇮🇳 CDSCO (India)")
            st.markdown(f"**License type:** `{cdsco_row['license_type']}`")
            st.markdown(f"**Timeline:** {cdsco_row['timeline_months']} months")
            st.markdown(f"**QMS required:** {cdsco_row['qms_required']}")
            st.markdown(f"**Clinical data:** {cdsco_row['clinical_data_required']}")
            st.markdown("**Checklist:**")
            for req in build_requirements(cdsco_row, "CDSCO"):
                st.markdown(f"- {req}")
        col_idx += 1

    if show_fda:
        with cols[col_idx]:
            st.markdown("#### 🇺🇸 FDA (USA)")
            st.markdown(f"**Pathway:** `{fda_row['pathway']}`")
            st.markdown(f"**Timeline:** {fda_row['timeline_months']} months")
            st.markdown(f"**Predicate needed:** {fda_row['predicate_needed']}")
            st.markdown(f"**Clinical trials:** {fda_row['clinical_trials_required']}")
            st.markdown("**Checklist:**")
            for req in build_requirements(fda_row, "FDA"):
                st.markdown(f"- {req}")
        col_idx += 1

    if show_eu:
        with cols[col_idx]:
            st.markdown("#### 🇪🇺 CE Mark (EU)")
            st.markdown(f"**Tech file:** `{eu_row['technical_file_type']}`")
            st.markdown(f"**Timeline:** {eu_row['timeline_months']} months")
            st.markdown(f"**Notified body:** {eu_row['notified_body_needed']}")
            st.markdown(f"**PMCF required:** {eu_row['pmcf_required']}")
            st.markdown("**Checklist:**")
            for req in build_requirements(eu_row, "EU"):
                st.markdown(f"- {req}")

    st.divider()
    st.caption("Data sources: CDSCO MDR 2017 | FDA 21 CFR | EU MDR 2017/745 | ISO 13485")
'''

# Write it to a file
with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written successfully!")

✅ app.py written successfully!


In [54]:
import subprocess, threading, time
from pyngrok import ngrok

# Attempt to disconnect any active tunnels before killing processes
# This addresses cases where ngrok.kill() might not immediately free up endpoints on the server side
try:
    active_tunnels = ngrok.get_tunnels()
    for tunnel in active_tunnels:
        print(f"Disconnecting active tunnel: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
except Exception as e:
    print(f"Error disconnecting tunnels (might be none active): {e}")

# Kill any existing ngrok processes to ensure a clean start
ngrok.kill()
time.sleep(5) # Add a longer delay to allow ngrok service to clean up

# Start Streamlit in the background
def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port=8501",
                    "--server.headless=true"])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Wait for it to start
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print("=" * 55)
print(f"  Your app is LIVE at: {public_url}")
print("=" * 55)
print("  Click the link above to open your app!")

Disconnecting active tunnel: https://judge-tactless-greedless.ngrok-free.dev
  Your app is LIVE at: NgrokTunnel: "https://judge-tactless-greedless.ngrok-free.dev" -> "http://localhost:8501"
  Click the link above to open your app!


In [55]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import google.generativeai as genai # Added for Gemini API
import json # Added for JSON parsing

st.set_page_config(
    page_title="MedTech Regulatory Navigator",
    page_icon="🧬",
    layout="wide"
)

# ── Custom CSS for professional look ────────────────────────────────────────
st.markdown("""
<style>
    .main { padding-top: 1rem; }
    .stMetric { background: #f8f9fa; border-radius: 8px; padding: 12px; }
    .risk-badge {
        display: inline-block; padding: 4px 12px;
        border-radius: 20px; font-size: 13px; font-weight: 600;
    }
    .footer { color: #888; font-size: 12px; text-align: center; margin-top: 2rem; }
</style>
""", unsafe_allow_html=True)

# ── Gemini API Setup ─────────────────────────────────────────────────────────
# For deployment on Streamlit Cloud, use st.secrets. For local, you might use os.environ.
# Make sure GEMINI_API_KEY is set in your Streamlit Cloud secrets.
# See: https://docs.streamlit.io/deploy/streamlit-cloud/connect-to-data-sources/secrets-management
if "GEMINI_API_KEY" in st.secrets:
    genai.configure(api_key=st.secrets["GEMINI_API_KEY"])
else:
    st.warning("GEMINI_API_KEY not found in Streamlit secrets. AI classification may not work.")
    genai.configure(api_key="") # Provide an empty key to avoid errors, but it won't work

model = genai.GenerativeModel("gemini-2.5-flash") # Changed model to gemini-2.5-flash

# ── AI Classification Function (from TpKKp9k00pwg) ───────────────────────────
def ai_classify_device(device_name, device_description=""):
    if not device_name:
        return None

    prompt = f"""You are a senior medical device regulatory affairs expert with
deep knowledge of CDSCO MDR 2017 (India), FDA 21 CFR (USA), and EU MDR 2017/745 (Europe).

Classify this medical device and return ONLY valid JSON, nothing else.

Device: {device_name}
{f"Description: {device_description}" if device_description else ""}

Return exactly this JSON structure:
{{
  "device_name": "{device_name}",
  "intended_use": "brief one-line intended use",
  "cdsco": {{
    "risk_class": "A or B or C or D",
    "license_type": "MD-5 or MD-9 or MD-14",
    "timeline_months": 0,
    "qms_required": "Yes or No",
    "clinical_data_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "fda": {{
    "risk_class": "I or II or III",
    "pathway": "Exempt or 510(k) or PMA",
    "predicate_needed": "No",
    "timeline_months": 0,
    "clinical_trials_required": "Yes or No",
    "ide_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "eu": {{
    "risk_class": "I or IIa or IIb or III",
    "notified_body_needed": "Yes or No",
    "timeline_months": 0,
    "technical_file_type": "Basic UDI-DI or Full Tech File",
    "clinical_evaluation_required": "Yes or No",
    "pmcf_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "confidence": "High or Medium or Low",
  "disclaimer": "any important caveats"
}}
Return ONLY the JSON object, no markdown, no explanation."""

    try:
        response = model.generate_content(prompt)
        raw = response.text.strip()

        # Clean markdown if present
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()
        return json.loads(raw)
    except Exception as e:
        st.error(f"AI classification failed: {e}. Please check your API key and try again.")
        return None

# ── Helpers ──────────────────────────────────────────────────────────────────
def get_risk_level(risk_class):
    return {"A":"Low","B":"Medium","C":"High","D":"Critical",
            "I":"Low","II":"Medium","III":"Critical",
            "IIa":"Medium","IIb":"High"}.get(risk_class,"Unknown")

def get_risk_color(level):
    return {"Low":"#1D9E75","Medium":"#BA7517",
            "High":"#D85A30","Critical":"#E24B4A"}.get(level,"#888")

# ── Sidebar ──────────────────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://img.icons8.com/color/96/dna-helix.png", width=60)
    st.title("Navigator")
    st.caption("MDR 2017 · FDA 21 CFR · EU MDR 2017/745")
    st.divider()

    st.subheader("Device Input")
    selected_device = st.text_input("Enter Device Name", "Smart Insulin Pen")
    device_description = st.text_area("Optional: Device Description",
                                    "Bluetooth-connected insulin delivery device that tracks doses and syncs with a smartphone app")

    st.subheader("Target Markets")
    show_cdsco = st.checkbox("🇮🇳 India (CDSCO)", value=True)
    show_fda   = st.checkbox("🇺🇸 USA (FDA)",     value=True)
    show_eu    = st.checkbox("🇪🇺 Europe (CE Mark)",value=True)

    st.divider()
    analyse_button = st.button("Analyse Device", type="primary", use_container_width=True)
    st.caption("Enter device details and click Analyse.")

    st.divider()
    st.subheader("About this tool")
    st.caption("""
    Encodes 3 major regulatory frameworks
    into a single decision-support tool.
    Reduces pathway scoping from hours to
    minutes. Built with Python + Streamlit +
    Gemini API.
    """)

# ── Header ───────────────────────────────────────────────────────────────────
st.markdown("## 🧬 MedTech Regulatory Pathway Navigator")
st.markdown(
    "Instantly identify the optimal regulatory pathway for any medical device "
    "across **CDSCO (India)**, **FDA 510(k)/PMA (USA)**, and **CE Mark (EU MDR)** using AI."
)
st.divider()

# Perform AI classification only when the button is clicked
if analyse_button:
    ai_classification = ai_classify_device(selected_device, device_description)

    if ai_classification:
        st.session_state['ai_classification_result'] = ai_classification
    else:
        st.session_state['ai_classification_result'] = None

# Display results if available in session state
if 'ai_classification_result' in st.session_state and st.session_state['ai_classification_result'] is not None:
    ai_result = st.session_state['ai_classification_result']

    # ── Intended use info box ──────────────────────────────────────────────
    st.info(f"**{ai_result['device_name']}** — Intended use: *{ai_result['intended_use']}*")

    # ── Risk classification ────────────────────────────────────────────────
    st.subheader("Risk Classification")
    col1, col2, col3 = st.columns(3)

    # CDSCO
    cdsco_class = ai_result['cdsco']['risk_class']
    cdsco_lvl   = get_risk_level(cdsco_class)
    cdsco_color = get_risk_color(cdsco_lvl)
    with col1:
        st.metric("CDSCO Class (India)", cdsco_class)
        st.markdown(
            f"<span class='risk-badge' style='background:{cdsco_color}22;color:{cdsco_color}'>{cdsco_lvl} Risk</span>",
            unsafe_allow_html=True
        )

    # FDA
    fda_class = ai_result['fda']['risk_class']
    fda_lvl   = get_risk_level(fda_class)
    fda_color = get_risk_color(fda_lvl)
    with col2:
        st.metric("FDA Class (USA)", fda_class)
        st.markdown(
            f"<span class='risk-badge' style='background:{fda_color}22;color:{fda_color}'>{fda_lvl} Risk</span>",
            unsafe_allow_html=True
        )

    # EU
    eu_class = ai_result['eu']['risk_class']
    eu_lvl   = get_risk_level(eu_class)
    eu_color = get_risk_color(eu_lvl)
    with col3:
        st.metric("EU MDR Class (Europe)", eu_class)
        st.markdown(
            f"<span class='risk-badge' style='background:{eu_color}22;color:{eu_color}'>{eu_lvl} Risk</span>",
            unsafe_allow_html=True
        )

    st.divider()

    # ── Timeline comparison chart ──────────────────────────────────────────
    st.subheader("Approval Timeline Comparison")

    markets, months, colors = [], [], []
    cmap = {"CDSCO (India)":"#1D9E75","FDA (USA)":"#378ADD","CE Mark (EU)":"#D85A30"}

    if show_cdsco:
        markets.append("CDSCO (India)")
        months.append(ai_result['cdsco']['timeline_months'])
        colors.append(cmap["CDSCO (India)"])
    if show_fda:
        markets.append("FDA (USA)")
        months.append(ai_result['fda']['timeline_months'])
        colors.append(cmap["FDA (USA)"])
    if show_eu:
        markets.append("CE Mark (EU)")
        months.append(ai_result['eu']['timeline_months'])
        colors.append(cmap["CE Mark (EU)"])

    if markets:
        fig = go.Figure(go.Bar(
            x=markets, y=months,
            marker_color=colors,
            text=[f"{m} months" for m in months],
            textposition="outside"
        ))
        avg = sum(months) / len(months)
        fig.add_hline(y=avg, line_dash="dot", line_color="#888",
                      annotation_text=f"Avg: {avg:.0f} mo",
                      annotation_position="top right")
        fig.update_layout(
            yaxis_title="Months to approval",
            plot_bgcolor="white", height=380,
            margin=dict(t=30,b=20),
            yaxis=dict(gridcolor="#f0f0f0")
        )
        st.plotly_chart(fig, use_container_width=True)

        fastest = min(zip(markets, months), key=lambda x: x[1])
        slowest = max(zip(markets, months), key=lambda x: x[1])
        c1, c2 = st.columns(2)
        c1.success(f"Fastest entry: **{fastest[0]}** — {fastest[1]} months")
        c2.warning(f"Longest path: **{slowest[0]}** — {slowest[1]} months")

    st.divider()

    # ── Tabbed pathway details ────────────────────────────────────────────
    st.subheader("Detailed Pathway Requirements")

    active_tabs = []
    if show_cdsco: active_tabs.append("🇮🇳 CDSCO")
    if show_fda:   active_tabs.append("🇺🇸 FDA")
    if show_eu:    active_tabs.append("🇪🇺 CE Mark")

    if active_tabs:
        tabs = st.tabs(active_tabs)
        tab_idx = 0

        if show_cdsco:
            with tabs[tab_idx]:
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("**Key Details (AI-derived)**")
                    st.markdown(f"- License type: `{ai_result['cdsco']['license_type']}`")
                    st.markdown(f"- Timeline: **{ai_result['cdsco']['timeline_months']} months**")
                    st.markdown(f"- QMS required: {ai_result['cdsco']['qms_required']}")
                    st.markdown(f"- Clinical data: {ai_result['cdsco']['clinical_data_required']}")
                with c2:
                    st.markdown("**AI Reasoning for Classification**")
                    st.markdown(f"- {ai_result['cdsco']['reasoning']}")
            tab_idx += 1

        if show_fda:
            with tabs[tab_idx]:
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("**Key Details (AI-derived)**")
                    st.markdown(f"- Pathway: `{ai_result['fda']['pathway']}`")
                    st.markdown(f"- Timeline: **{ai_result['fda']['timeline_months']} months**")
                    st.markdown(f"- Predicate needed: {ai_result['fda']['predicate_needed']}")
                    st.markdown(f"- Clinical trials: {ai_result['fda']['clinical_trials_required']}")
                    st.markdown(f"- IDE required: {ai_result['fda']['ide_required']}")
                with c2:
                    st.markdown("**AI Reasoning for Classification**")
                    st.markdown(f"- {ai_result['fda']['reasoning']}")
            tab_idx += 1

        if show_eu:
            with tabs[tab_idx]:
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("**Key Details (AI-derived)**")
                    st.markdown(f"- Tech file: `{ai_result['eu']['technical_file_type']}`")
                    st.markdown(f"- Timeline: **{ai_result['eu']['timeline_months']} months**")
                    st.markdown(f"- Notified body: {ai_result['eu']['notified_body_needed']}")
                    st.markdown(f"- Clinical eval: {ai_result['eu']['clinical_evaluation_required']}")
                    st.markdown(f"- PMCF required: {ai_result['eu']['pmcf_required']}")
                with c2:
                    st.markdown("**AI Reasoning for Classification**")
                    st.markdown(f"- {ai_result['eu']['reasoning']}")

    st.divider()

    # ── Full comparison summary table ──────────────────────────────────────
    st.subheader("Side-by-Side Summary")

    summary_data = {
        "Parameter"      : ["Risk Class","Pathway/License","Timeline (months)",
                             "QMS / ISO 13485","Clinical Data","Notified Body / IDE"],
        "CDSCO (India)"  : [ai_result['cdsco']['risk_class'], ai_result['cdsco']['license_type'],
                             ai_result['cdsco']['timeline_months'], ai_result['cdsco']['qms_required'],
                             ai_result['cdsco']['clinical_data_required'], "N/A"], # AI does not provide direct IDE/NB info for CDSCO
        "FDA (USA)"      : [ai_result['fda']['risk_class'], ai_result['fda']['pathway'],
                             ai_result['fda']['timeline_months'], "Yes (21 CFR 820)", # General FDA requirement
                             ai_result['fda']['clinical_trials_required'], ai_result['fda']['ide_required']],
        "CE Mark (EU)"   : [ai_result['eu']['risk_class'], ai_result['eu']['technical_file_type'],
                             ai_result['eu']['timeline_months'], "Yes (ISO 13485)", # General EU requirement
                             ai_result['eu']['clinical_evaluation_required'], ai_result['eu']['notified_body_needed']]
    }
    st.dataframe(pd.DataFrame(summary_data), use_container_width=True, hide_index=True)

    st.divider()
    st.markdown(
        f"<div class=\'footer\'>AI Confidence: **{ai_result['confidence']}** | "
        f"Disclaimer: {ai_result['disclaimer']} | "
        "Built with Python + Streamlit + Plotly + Gemini API</div>",
        unsafe_allow_html=True
    )
else:
    st.info("Enter a medical device name and description in the sidebar and click 'Analyse Device' to get AI-powered regulatory insights.")

'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ Upgraded app.py written with AI classification!")
print("Now re-run your ngrok launch cell to see the new AI-powered version.")

✅ Upgraded app.py written with AI classification!
Now re-run your ngrok launch cell to see the new AI-powered version.


In [56]:
from google.colab import files

# Download the app
files.download("app.py")

# Download the Excel data
files.download("data/regulatory_rules.xlsx")

print("✅ Files downloaded to your computer!")
print("Keep these safe — they're your complete project.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded to your computer!
Keep these safe — they're your complete project.


In [57]:
!pip install anthropic -q

import anthropic
import json
from google.colab import userdata

# Get your free API key from: https://console.anthropic.com
# Usage is free up to generous limits for testing
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✅ Anthropic client ready!")

✅ Anthropic client ready!


In [58]:
# Replace anthropic with google's free API
!pip install google-generativeai -q

import google.generativeai as genai
import json
from google.colab import userdata

# Get free API key from: https://aistudio.google.com/app/apikey
# Add your GEMINI_API_KEY to Colab secrets (the '🔑' icon on the left sidebar)
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")  # Changed model to gemini-2.5-flash

def ai_classify_device(device_name, device_description=""):
    prompt = f"""You are a senior medical device regulatory affairs expert with
deep knowledge of CDSCO MDR 2017 (India), FDA 21 CFR (USA), and EU MDR 2017/745 (Europe).

Classify this medical device and return ONLY valid JSON, nothing else.

Device: {device_name}
{f"Description: {device_description}" if device_description else ""}

Return exactly this JSON structure:
{{
  "device_name": "{device_name}",
  "intended_use": "brief one-line intended use",
  "cdsco": {{
    "risk_class": "A or B or C or D",
    "license_type": "MD-5 or MD-9 or MD-14",
    "timeline_months": 0,
    "qms_required": "Yes or No",
    "clinical_data_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "fda": {{
    "risk_class": "I or II or III",
    "pathway": "Exempt or 510(k) or PMA",
    "predicate_needed": "Yes or No",
    "timeline_months": 0,
    "clinical_trials_required": "Yes or No",
    "ide_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "eu": {{
    "risk_class": "I or IIa or IIb or III",
    "notified_body_needed": "Yes or No",
    "timeline_months": 0,
    "technical_file_type": "Basic UDI-DI or Full Tech File",
    "clinical_evaluation_required": "Yes or No",
    "pmcf_required": "Yes or No",
    "reasoning": "one sentence why this class"
  }},
  "confidence": "High or Medium or Low",
  "disclaimer": "any important caveats"
}}
Return ONLY the JSON object, no markdown, no explanation."""

    response = model.generate_content(prompt)
    raw = response.text.strip()

    # Clean markdown if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    return json.loads(raw)


# Test it
print("Testing with 'Smart Insulin Pen'...")
test = ai_classify_device(
    "Smart Insulin Pen",
    "Bluetooth-connected insulin delivery device that tracks doses and syncs with a smartphone app"
)
print(f"Device      : {test['device_name']}")
print(f"Intended use: {test['intended_use']}")
print(f"\nCDSCO : Class {test['cdsco']['risk_class']} | {test['cdsco']['timeline_months']} months")
print(f"  Why? {test['cdsco']['reasoning']}")
print(f"\nFDA   : Class {test['fda']['risk_class']} | {test['fda']['pathway']} | {test['fda']['timeline_months']} months")
print(f"  Why? {test['fda']['reasoning']}")
print(f"\nEU    : Class {test['eu']['risk_class']} | {test['eu']['timeline_months']} months")
print(f"  Why? {test['eu']['reasoning']}")
print(f"\nConfidence: {test['confidence']}")
print("✅ AI classifier working!")

Testing with 'Smart Insulin Pen'...
Device      : Smart Insulin Pen
Intended use: A device intended for the subcutaneous self-administration of insulin by persons with diabetes, enabling tracking of doses and synchronization with a smartphone application for record-keeping and management.

CDSCO : Class C | 12 months
  Why? This is an active device for the therapeutic administration of a critical substance (insulin) where an error could lead to serious harm, thus falling into Class C.

FDA   : Class II | 510(k) | 8 months
  Why? Similar to existing Class II smart insulin pens (e.g., product code FPT), this device shares the same intended use and technological characteristics, allowing for 510(k) clearance.

EU    : Class IIb | 15 months
  Why? An active device intended to administer a critical substance (insulin) in a potentially hazardous manner, as per Annex VIII Rule 2.2, thus classified as Class IIb.

Confidence: High
✅ AI classifier working!


In [59]:
import google.generativeai as genai

print("Available Gemini Models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"- {m.name} (Supports generateContent)")
    else:
        print(f"- {m.name}")

Available Gemini Models:
- models/gemini-2.5-flash (Supports generateContent)
- models/gemini-2.5-pro (Supports generateContent)
- models/gemini-2.0-flash (Supports generateContent)
- models/gemini-2.0-flash-001 (Supports generateContent)
- models/gemini-2.0-flash-lite-001 (Supports generateContent)
- models/gemini-2.0-flash-lite (Supports generateContent)
- models/gemini-2.5-flash-preview-tts (Supports generateContent)
- models/gemini-2.5-pro-preview-tts (Supports generateContent)
- models/gemma-4-26b-a4b-it (Supports generateContent)
- models/gemma-4-31b-it (Supports generateContent)
- models/gemini-flash-latest (Supports generateContent)
- models/gemini-flash-lite-latest (Supports generateContent)
- models/gemini-pro-latest (Supports generateContent)
- models/gemini-2.5-flash-lite (Supports generateContent)
- models/gemini-2.5-flash-image (Supports generateContent)
- models/gemini-3-pro-preview (Supports generateContent)
- models/gemini-3-flash-preview (Supports generateContent)
- m

In [60]:
# Replace these two lines in Cell 11's app_code string:
# import anthropic
# client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# With these:
import google.generativeai as genai
import os
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

In [61]:
# First, create a clean requirements.txt file
with open("requirements.txt", "w") as f:
    f.write("""streamlit
pandas
plotly
openpyxl
google-generativeai
""")

# Create a .gitignore so you don't accidentally upload your API key
with open(".gitignore", "w") as f:
    f.write("""venv/
__pycache__/
*.pyc
.env
secrets.toml
""")

# Create a secrets template (you'll fill this on Streamlit Cloud)
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets_template.toml", "w") as f:
    f.write('GEMINI_API_KEY = "paste-your-key-here"\n')

print("✅ All deployment files created!")

# Now download everything
from google.colab import files
files.download("app.py")
files.download("requirements.txt")
files.download(".gitignore")
files.download("data/regulatory_rules.xlsx")
print("✅ Files downloading to your computer!")

✅ All deployment files created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloading to your computer!
